# Notebook 2: Mechanical Systems — Harmonic Oscillators & Simple Pendulum

## Course: AP Physics C & Calculus BC Advanced Mechanics

### Objectives:
1. Derive the equations of motion for a **Simple Pendulum** using Lagrangian Mechanics.
2. Contrast the **generalized coordinate** $\theta$ approach with Newtonian free-body diagrams.
3. Simulate exact non-linear pendulum dynamics using `scipy.integrate.solve_ivp`.
4. Compare exact non-linear oscillations with the Calculus BC small-angle approximation ($\sin\theta \approx \theta$).


In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

sp.init_printing()


---
## 1. Symbolic Derivation of the Simple Pendulum

A pendulum mass $m$ is suspended by a rigid rod of length $l$ in a gravitational field $g$.
- **Position**: $x = l \sin\theta$, $y = -l \cos\theta$
- **Velocity**: $v^2 = \dot{x}^2 + \dot{y}^2 = l^2 \dot{\theta}^2$
- **Kinetic Energy**: $T = \frac{1}{2} m l^2 \dot{\theta}^2$
- **Potential Energy**: $V = -m g l \cos\theta$
- **Lagrangian**: $L = T - V = \frac{1}{2} m l^2 \dot{\theta}^2 + m g l \cos\theta$


In [ ]:
t = sp.Symbol('t', real=True)
m, g, l = sp.symbols('m g l', positive=True)
theta = sp.Function('theta')(t)
omega = sp.diff(theta, t)

# Kinetic and Potential Energy
T = sp.Rational(1, 2) * m * l**2 * omega**2
V = -m * g * l * sp.cos(theta)
L = T - V

# Euler-Lagrange Equation
dL_domega = sp.diff(L, omega)
dt_dL_domega = sp.diff(dL_domega, t)
dL_dtheta = sp.diff(L, theta)

el_eq = sp.simplify(dt_dL_domega - dL_dtheta)
eq = sp.Eq(el_eq / (m * l**2), 0)
print('Derived Equation of Motion for Simple Pendulum:')
sp.pprint(eq)


---
## 2. Numerical Simulation & Comparison with Small-Angle Approximation

We solve the exact non-linear system:
$$\ddot{\theta} + \frac{g}{l} \sin\theta = 0$$

versus the linear small-angle system:
$$\ddot{\theta} + \frac{g}{l} \theta = 0$$


In [ ]:
# Parameters
g_val = 9.81
l_val = 1.0
w0_sq = g_val / l_val

# System of First-Order ODEs
def pendulum_ode(t, y):
    theta, omega = y
    dtheta_dt = omega
    domega_dt = -w0_sq * np.sin(theta)
    return [dtheta_dt, domega_dt]

# Initial Conditions: Large angle (120 degrees = 2.094 rad) to highlight non-linearity
theta0 = np.radians(120.0)
y0 = [theta0, 0.0]
t_span = (0, 10)
t_eval = np.linspace(0, 10, 1000)

# Solve ODE
sol = solve_ivp(pendulum_ode, t_span, y0, t_eval=t_eval)

# Small angle linear solution for comparison
theta_linear = theta0 * np.cos(np.sqrt(w0_sq) * t_eval)

# Plotting
plt.figure(figsize=(10, 5))
plt.plot(sol.t, np.degrees(sol.y[0]), 'b-', label='Exact Non-Linear (Lagrangian)')
plt.plot(t_eval, np.degrees(theta_linear), 'r--', label='Small-Angle Approx (Linear)')
plt.title('Simple Pendulum Motion: Large Initial Angle (120°)', fontsize=14)
plt.xlabel('Time (s)', fontsize=12)
plt.ylabel('Angle θ (degrees)', fontsize=12)
plt.grid(True)
plt.legend(fontsize=11)
plt.savefig('/workspace/scratch/pendulum_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Pendulum plot generated successfully!')
